### Day 1 Full load

In [0]:
data_day1 = [
    (1, "C001", "Laptop", 50000, "2024-01-01"),
    (2, "C002", "Mobile", 15000, "2024-01-02"),
    (3, "C003", "Tablet", 20000, "2024-01-03"),
    (4, "C004", "Laptop", 55000, "2024-01-04"),
    (5, "C005", "Headphones", 3000, "2024-01-05")
]

columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

df_day1 = spark.createDataFrame(data_day1, columns)


In [0]:
df_day1.display()

### STEP 1: FULL LOAD (ONLY ONCE)

In [0]:
df_day1.write \
    .mode("overwrite") \
    .parquet("/Volumes/workspace/default/day2_week2")

### INCREMENTAL LOAD PIPELINE

### Read Existing Data

In [0]:
existing_df = spark.read.parquet("/Volumes/workspace/default/day2_week2")
existing_df.display()

### Get last_loaded_date (DYNAMIC)

In [0]:
from pyspark.sql.functions import max

last_loaded_date = existing_df.select(
    max("updated_date")
).collect()[0][0]

print("Last Loaded Date:", last_loaded_date)

### Day 2 Incremental Load

In [0]:
data_day2 = [
    (3, "C003", "Tablet", 22000, "2024-01-06"),  # updated
    (4, "C004", "Laptop", 56000, "2024-01-06"),  # updated
    (6, "C006", "Camera", 30000, "2024-01-06"),  # new
    (7, "C007", "Mobile", 18000, "2024-01-07"),  # new
    (8, "C008", "Watch", 8000, "2024-01-07")     # new
]

df_day2 = spark.createDataFrame(data_day2, columns)

In [0]:
df_day2.display()

### Read New Source Data

In [0]:
new_df = df_day2
new_df.display()

### Filter Incremental Records

In [0]:
from pyspark.sql.functions import col

incremental_df = new_df.filter(
    col("updated_date") > last_loaded_date
)

### What happens here?
### If last_loaded_date = "2024-01-05"
### You get:
### 2024-01-06
### 2024-01-07
### Only new + updated records

### Handle Updates (UPSERT LOGIC)

In [0]:
filtered_existing = existing_df.join(
    incremental_df,
    on="order_id",
    how="left_anti"
)
# Remove older versions of order_id 3 and 4

### Combine Data

In [0]:
final_df = filtered_existing.union(incremental_df)
final_df.display()

In [0]:
%python
final_df.write \
    .mode("overwrite") \
    .parquet("/Volumes/workspace/default/day2_week2")
